# Big Data Workbook

## Lab 1: Working with the PySpark Shell

In [3]:
sc

<SparkContext master=local[*] appName=PySparkShell>

In [4]:
spark

## Lab 2: Basic Python

In [4]:
myrdd = sc.textFile("file:/home/student/Data/kv1.txt")
myrdd.take(5)

['238\x01val_238',
 '86\x01val_86',
 '311\x01val_311',
 '27\x01val_27',
 '165\x01val_165']

In [5]:
myrdd = sc.textFile("file:/home/student/Data/kv1.txt").map(lambda line: line.split("\x01"))
myrdd.take(5)

[['238', 'val_238'],
 ['86', 'val_86'],
 ['311', 'val_311'],
 ['27', 'val_27'],
 ['165', 'val_165']]

In [6]:
def replaceStr(string, srch, repl):
    return string.replace(srch, repl)

myrdd = sc.textFile("file:/home/student/Data/kv1.txt").map(lambda line: line.split("\x01")).map(lambda array: replaceStr(array[1], "val_", "value is "))
myrdd.take(5)

['value is 238', 'value is 86', 'value is 311', 'value is 27', 'value is 165']

In [7]:
def replaceStr(string, srch, repl):
    return string.replace(srch, repl)

myrdd = sc.textFile("file:/home/student/Data/kv1.txt").map(lambda line: line.split("\x01")).map(lambda array: replaceStr(array[1], "val_", "value is ")).map(lambda string: string[9:])
myrdd.take(5)

['238', '86', '311', '27', '165']

In [8]:
def replaceStr(string, srch, repl):
    return string.replace(srch, repl)

myrdd = sc.textFile("file:/home/student/Data/kv1.txt").map(lambda line: line.split("\x01")).map(lambda array: replaceStr(array[1], "val_", "value is ")).map(lambda string: string[9:]).map(lambda string: (int(string), string))
myrdd.take(5)

[(238, '238'), (86, '86'), (311, '311'), (27, '27'), (165, '165')]

In [9]:
def evenNum(num):
    if (num%2 == 0): return True
    else: return False
    
evenRDD = myrdd.filter(lambda tupl: evenNum(tupl[0]))
evenRDD.take(5)

[(238, '238'), (86, '86'), (278, '278'), (98, '98'), (484, '484')]

In [10]:
evenRDD = myrdd.filter(lambda tupl: not (tupl[0]%2))
evenRDD.take(5)

[(238, '238'), (86, '86'), (278, '278'), (98, '98'), (484, '484')]

In [11]:
def prTuple(coll):
    for tupl in coll:
        for item in tupl:
            if type(item) is int:
                print(item+1000)
            elif type(item) is str:
                print("\t", item*2)
            else:
                print("Not int nor string")

small_list = evenRDD.take(5)
prTuple(small_list)

1238
	 238238
1086
	 8686
1278
	 278278
1098
	 9898
1484
	 484484


In [12]:
aliceRDD = sc.textFile("file:/home/student/Data/alice_in_wonderland.txt")
def makeCap(string):
    capWords = " "
    for word in string.split(" "):
        capWords += (word[0:1].upper() + word[1:])
        capWords += " "
    return capWords

print(makeCap("test this string to see all words in cap"))

 Test This String To See All Words In Cap 


In [13]:
capRDD = aliceRDD.map(makeCap)
capRDD.take(5)

[' The Project Gutenberg EBook Of Alice’s Adventures In Wonderland, By Lewis Carroll ',
 '  ',
 ' This EBook Is For The Use Of Anyone Anywhere In The United States And ',
 ' Most Other Parts Of The World At No Cost And With Almost No Restrictions ',
 ' Whatsoever. You May Copy It, Give It Away Or Re-use It Under The Terms ']

In [14]:
def capWords2(line):
    result=""
    for word in line.split(" "):
        result = result + word.capitalize() + " "
    return result

aliceRDD3 = aliceRDD.map(capWords2)
aliceRDD3.take(5)

['The Project Gutenberg Ebook Of Alice’s Adventures In Wonderland, By Lewis Carroll ',
 ' ',
 'This Ebook Is For The Use Of Anyone Anywhere In The United States And ',
 'Most Other Parts Of The World At No Cost And With Almost No Restrictions ',
 'Whatsoever. You May Copy It, Give It Away Or Re-use It Under The Terms ']

In [15]:
def inverseCap(line):
    result = ""
    for word in line.split(" "):
        if len(word) > 1:
            result = result + word[0].lower() + word[1:].upper() + " "
        elif len(word) == 1:
            result = result + word[0].lower() + " "
        else:
            result = result + " "
    return result
aliceRDD4 = aliceRDD.map(inverseCap)
aliceRDD4.take(5)

['tHE pROJECT gUTENBERG eBOOK oF aLICE’S aDVENTURES iN wONDERLAND, bY lEWIS cARROLL ',
 ' ',
 'tHIS eBOOK iS fOR tHE uSE oF aNYONE aNYWHERE iN tHE uNITED sTATES aND ',
 'mOST oTHER pARTS oF tHE wORLD aT nO cOST aND wITH aLMOST nO rESTRICTIONS ',
 'wHATSOEVER. yOU mAY cOPY iT, gIVE iT aWAY oR rE-USE iT uNDER tHE tERMS ']

## Lab 3: Working with Core API Transformations

In [17]:
aliceRDD = sc.textFile("alice.txt")
aliceRDD.take(5)

['The Project Gutenberg eBook of Alice’s Adventures in Wonderland, by Lewis Carroll',
 '',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms']

In [18]:
jsonRDD = sc.wholeTextFiles("/user/student/json_files")
jsonRDD.take(1)

[('hdfs://localhost:9000/user/student/json_files/1.json',
  '[\n\t{\n\t\t"id": 1955,\n\t\t"cust_since": "2022-06-04",\n\t\t"phone_model": "Galaxy s21 Ultra"\n\t},\n\t{\n\t\t"id": 317,\n\t\t"cust_since": "2019-09-04",\n\t\t"phone_model": "Galaxy Z Fold3"\n\t},\n\t{\n\t\t"id": 101,\n\t\t"cust_since": "2019-01-02",\n\t\t"phone_model": "Galaxy A52s"\n\t},\n\t{\n\t\t"id": 5265,\n\t\t"cust_since": "2020-04-11",\n\t\t"phone_model": "Galaxy A52s"\n\t},\n\t{\n\t\t"id": 6219,\n\t\t"cust_since": "2019-06-15",\n\t\t"phone_model": "Galaxy s21 Ultra"\n\t},\n\t{\n\t\t"id": 1892,\n\t\t"cust_since": "2019-01-01",\n\t\t"phone_model": "Galaxy A12"\n\t}\n]')]

In [19]:
import json

In [20]:
myrdd = jsonRDD.map(lambda tup: json.loads(tup[1]))
myrdd.take(2)

[[{'id': 1955, 'cust_since': '2022-06-04', 'phone_model': 'Galaxy s21 Ultra'},
  {'id': 317, 'cust_since': '2019-09-04', 'phone_model': 'Galaxy Z Fold3'},
  {'id': 101, 'cust_since': '2019-01-02', 'phone_model': 'Galaxy A52s'},
  {'id': 5265, 'cust_since': '2020-04-11', 'phone_model': 'Galaxy A52s'},
  {'id': 6219, 'cust_since': '2019-06-15', 'phone_model': 'Galaxy s21 Ultra'},
  {'id': 1892, 'cust_since': '2019-01-01', 'phone_model': 'Galaxy A12'}],
 [{'id': 5449, 'cust_since': '2021-10-05', 'phone_model': 'Galaxy S10'},
  {'id': 1046, 'cust_since': '2021-06-23', 'phone_model': 'Galaxy Z Fold3'},
  {'id': 8295, 'cust_since': '2022-01-11', 'phone_model': 'Galaxy s21 Ultra'},
  {'id': 8549, 'cust_since': '2019-12-25', 'phone_model': 'Galaxy S8'},
  {'id': 3913, 'cust_since': '2020-10-22', 'phone_model': 'Galaxy A52s'},
  {'id': 1450, 'cust_since': '2022-06-20', 'phone_model': 'Galaxy A03'},
  {'id': 7184, 'cust_since': '2021-07-10', 'phone_model': 'Galaxy Note20'},
  {'id': 2070, 'cust_

In [21]:
import json
myrdd = jsonRDD.flatMap(lambda tup: json.loads(tup[1])).map(lambda  kv: (kv.get("id", None),(kv.get("cust_since", None), (kv.get("phone_model", None)))))
myrdd.take(5)

[(1955, ('2022-06-04', 'Galaxy s21 Ultra')),
 (317, ('2019-09-04', 'Galaxy Z Fold3')),
 (101, ('2019-01-02', 'Galaxy A52s')),
 (5265, ('2020-04-11', 'Galaxy A52s')),
 (6219, ('2019-06-15', 'Galaxy s21 Ultra'))]

In [22]:
words = aliceRDD.flatMap(lambda line: line.split(" ")).count()
print(words)

31192


In [23]:
distinct_words = aliceRDD.flatMap(lambda line: line.split(" ")).distinct().count()
print(distinct_words)

5981


In [24]:
fruit1 = ["Banana", "Pear", "Kiwi", "Peach", "Grape"]
fruit1RDD = sc.parallelize(fruit1)
fruit1RDD.collect()

['Banana', 'Pear', 'Kiwi', 'Peach', 'Grape']

In [25]:
fruit2 = ["Strawberry", "Kiwi", "Watermelon", "Banana", "Apple"]
fruit2RDD = sc.parallelize(fruit2)
fruit2RDD.collect()

['Strawberry', 'Kiwi', 'Watermelon', 'Banana', 'Apple']

In [26]:
unionRDD = fruit1RDD.union(fruit2RDD)
unionRDD.collect()

['Banana',
 'Pear',
 'Kiwi',
 'Peach',
 'Grape',
 'Strawberry',
 'Kiwi',
 'Watermelon',
 'Banana',
 'Apple']

In [27]:
intersectRDD = fruit1RDD.intersection(fruit2RDD)
intersectRDD.collect()

['Kiwi', 'Banana']

In [28]:
subtractRDD = fruit1RDD.subtract(fruit2RDD)
subtractRDD.collect()

['Pear', 'Peach', 'Grape']

In [29]:
cartesianRDD = fruit1RDD.cartesian(fruit2RDD)
cartesianRDD.collect()

[('Banana', 'Strawberry'),
 ('Banana', 'Kiwi'),
 ('Banana', 'Watermelon'),
 ('Banana', 'Banana'),
 ('Banana', 'Apple'),
 ('Pear', 'Strawberry'),
 ('Pear', 'Kiwi'),
 ('Pear', 'Watermelon'),
 ('Pear', 'Banana'),
 ('Pear', 'Apple'),
 ('Kiwi', 'Strawberry'),
 ('Kiwi', 'Kiwi'),
 ('Kiwi', 'Watermelon'),
 ('Kiwi', 'Banana'),
 ('Kiwi', 'Apple'),
 ('Peach', 'Strawberry'),
 ('Peach', 'Kiwi'),
 ('Peach', 'Watermelon'),
 ('Peach', 'Banana'),
 ('Peach', 'Apple'),
 ('Grape', 'Strawberry'),
 ('Grape', 'Kiwi'),
 ('Grape', 'Watermelon'),
 ('Grape', 'Banana'),
 ('Grape', 'Apple')]

In [30]:
def perPartition(it):
    print("I just did a heavy operation once per partition")
    for item in it:
        yield "updated_" + item

In [31]:
my_records = ["record1", "record2", "record3", "record4", "record5"]
recordsRDD = sc.parallelize(my_records,2).mapPartitions(lambda iterator: perPartition(iterator))
recordsRDD.collect()

I just did a heavy operation once per partition
I just did a heavy operation once per partition


['updated_record1',
 'updated_record2',
 'updated_record3',
 'updated_record4',
 'updated_record5']

In [1]:
def perPartitionIndex(index, it):
    print("I just did a heavy operation in partition:", index)
    for item in it:
        yield "updated_" + str(index) + "_" + item

In [33]:
my_records = ["record1", "record2", "record3", "record4", "record5"]
records2RDD = sc.parallelize(my_records,2).mapPartitionsWithIndex(lambda index, iterator: perPartitionIndex(index, iterator))
records2RDD.collect()

I just did a heavy operation in partition: 0
I just did a heavy operation in partition: 1


['updated_0_record1',
 'updated_0_record2',
 'updated_1_record3',
 'updated_1_record4',
 'updated_1_record5']

In [34]:
def actionPerPartition1(it):
    print("I just did a foreachPartition action")

In [35]:
recordsRDD.foreachPartition(actionPerPartition1)

I just did a foreachPartition action
I just did a foreachPartition action===>                            (1 + 1) / 2]


In [36]:
def actionPerPartition2(it):
    print("I have", len(list(it)), "elements")

In [37]:
recordsRDD.foreachPartition(actionPerPartition2)

I just did a heavy operation once per partition                     (0 + 1) / 2]
I have 2 elements
I just did a heavy operation once per partition                     (1 + 1) / 2]
I have 3 elements


In [38]:
records2RDD.foreachPartition(actionPerPartition2)

I just did a heavy operation in partition: 0
I have 2 elements
I just did a heavy operation in partition: 1
I have 3 elements


In [39]:
def printElements(iterator):
    for item in iterator: print(item)
    print("*********************")

In [40]:
orderNums = [1,2,3,4,5,6,7,8,9]
orderRDD = sc.parallelize(orderNums, 4)
print("Number of partitions: ", orderRDD.getNumPartitions())
orderRDD.foreachPartition(printElements)

Number of partitions:  4


1
2
*********************
3
4
*********************
5Stage 31:==============>                                           (1 + 1) / 4]
6
*********************
7Stage 31:===========================================>              (3 + 1) / 4]
8
9
*********************


In [41]:
unitRDD = orderRDD.coalesce(2)
print("Number of partitions: ", unitRDD.getNumPartitions())
unitRDD.foreachPartition(printElements)

Number of partitions:  2


1
2
3
4
*********************
5Stage 32:=============================>                            (1 + 1) / 2]
6
7
8
9
*********************


In [42]:
repartRDD = orderRDD.repartition(4)
print("Number of partitions: ", repartRDD.getNumPartitions())
repartRDD.foreachPartition(printElements)

Number of partitions:  4


5Stage 34:>                                                         (0 + 1) / 4]
6
7
8
9
*********************
3Stage 34:==============>                                           (1 + 1) / 4]
4
*********************
1
2
*********************
*********************================================>              (3 + 1) / 4]


In [43]:
import random
seed = int(random.random())
to100RD = sc.parallelize(range(100))
print(to100RD.sample(False, 0.2, seed).collect())

[14, 24, 29, 35, 41, 45, 59, 60, 64, 86, 93, 98, 99]


In [44]:
to100RD = sc.parallelize(range(100))
print(to100RD.sample(True, 0.2, 3654).collect())

[0, 1, 1, 12, 25, 26, 32, 35, 37, 41, 42, 54, 57, 78, 81, 85, 87, 90]


## Lab 4: Working with Pair RDDs

In [46]:
mydata1 = ["Henry, 42, M", "Jessica, 16, F",
"Sharon, 21, F", "Jonathan, 27, M",
"Shaun, 11, M", "Jasmine, 62, F"]

In [47]:
myrdd = sc.parallelize(mydata1).map(lambda line: line.split(",")).keyBy(lambda collection: collection[0])
myrdd.take(5)

[('Henry', ['Henry', ' 42', ' M']),
 ('Jessica', ['Jessica', ' 16', ' F']),
 ('Sharon', ['Sharon', ' 21', ' F']),
 ('Jonathan', ['Jonathan', ' 27', ' M']),
 ('Shaun', ['Shaun', ' 11', ' M'])]

In [48]:
myrdd = sc.parallelize(mydata1).map(lambda line:line.split(",")).map(lambda collection: (collection[0], collection))
myrdd.take(5)

[('Henry', ['Henry', ' 42', ' M']),
 ('Jessica', ['Jessica', ' 16', ' F']),
 ('Sharon', ['Sharon', ' 21', ' F']),
 ('Jonathan', ['Jonathan', ' 27', ' M']),
 ('Shaun', ['Shaun', ' 11', ' M'])]

In [49]:
myrdd = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (collection[0], (int(collection[1]), collection[2])))
myrdd.take(5)

[('Henry', (42, ' M')),
 ('Jessica', (16, ' F')),
 ('Sharon', (21, ' F')),
 ('Jonathan', (27, ' M')),
 ('Shaun', (11, ' M'))]

In [50]:
mydata2 = ["Henry,red:blue",
"Jessica,pink:turquoise",
"Sharon,blue:pink",
"Jonathan,blue:green",
"Shaun,sky blue:red",
"Jasmine,yellow:orange"]

In [51]:
favColorRDD = sc.parallelize(mydata2).map(lambda line: line.split(",")).map(lambda collection: (collection[0], collection[1])).flatMapValues(lambda colors: colors.split(":"))
favColorRDD.take(5)

[('Henry', 'red'),
 ('Henry', 'blue'),
 ('Jessica', 'pink'),
 ('Jessica', 'turquoise'),
 ('Sharon', 'blue')]

In [52]:
genderSumAge = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (collection[2], int(collection[1])))
genderSumAge.take(6)

[(' M', 42), (' F', 16), (' F', 21), (' M', 27), (' M', 11), (' F', 62)]

In [53]:
genderSumAge = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (collection[2], int(collection[1]))).reduceByKey(lambda v1, v2: v1+v2)
genderSumAge.collect()

[(' M', 80), (' F', 99)]

In [54]:
genderMaxAge = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (collection[2], int(collection[1]))).reduceByKey(lambda v1, v2: max(v1,v2))
genderMaxAge.take(6)

[(' M', 42), (' F', 62)]

In [55]:
genderMinAge = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (collection[2], int(collection[1]))).reduceByKey(lambda v1, v2: min(v1,v2))
genderMinAge.take(6)

[(' M', 11), (' F', 16)]

In [56]:
countGender = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (collection[2], int(collection[1]))).countByKey()
print(countGender)

defaultdict(<class 'int'>, {' M': 3, ' F': 3})


In [57]:
sortAgeRDD = sc.parallelize(mydata1).map(lambda line: line.split(",")).map(lambda collection: (int(collection[1]), collection[0])).sortByKey(ascending=False).map(lambda tup: (tup[1], tup[0]))
sortAgeRDD.take(3)

[('Jasmine', 62), ('Henry', 42), ('Jonathan', 27)]

In [58]:
colorLikers = sc.parallelize(mydata2).map(lambda line: line.split(",")).map(lambda collection: (collection[0], collection[1])).flatMapValues(lambda colors: colors.split(":")).map(lambda tup: (tup[1], tup[0])).groupByKey()

for color in colorLikers.collect():
    print(color[0])
    for person in color[1]:
        print(" ", person)

red
  Henry
  Shaun
blue
  Henry
  Sharon
  Jonathan
pink
  Jessica
  Sharon
turquoise
  Jessica
green
  Jonathan
sky blue
  Shaun
yellow
  Jasmine
orange
  Jasmine


In [59]:
zeroValue = []

def seqOp(accumulator, element):
    accumulator.append(element)
    return accumulator

def combOp(accumulator1, accumulator2):
    return accumulator1 + accumulator2

colorLikers2 = sc.parallelize(mydata2,2).map(lambda line: line.split(",")).map(lambda collection: (collection[0], collection[1])).flatMapValues(lambda colors: colors.split(":")).map(lambda tup: (tup[1], tup[0])).aggregateByKey(zeroValue, seqOp, combOp)

colorLikers2.collect()

[('green', ['Jonathan']),
 ('sky blue', ['Shaun']),
 ('yellow', ['Jasmine']),
 ('orange', ['Jasmine']),
 ('red', ['Henry', 'Shaun']),
 ('blue', ['Henry', 'Sharon', 'Jonathan']),
 ('pink', ['Jessica', 'Sharon']),
 ('turquoise', ['Jessica'])]

In [61]:
data1RDD = sc.parallelize(mydata1).map(lambda line: (line.split(",")[0], [line.split(",")[1], line.split(",")[2]]))

data2RDD = sc.parallelize(mydata2).map(lambda line: (line.split(",")[0], line.split(",")[1])).map(lambda tup: (tup[0], [tup[1].split(":")[0], tup[1].split(":")[1]]))

joinRDD = data1RDD.join(data2RDD)

outRDD = joinRDD.map(lambda tup: [tup[0], tup[1][0][0], tup[1][0][1], tup[1][1][0], tup[1][1][1]]).map(lambda lst: ",".join(lst))

outRDD.saveAsTextFile("/user/student/persons/")
outRDD.take(5)

['Jessica, 16, F,pink,turquoise',
 'Sharon, 21, F,blue,pink',
 'Jonathan, 27, M,blue,green',
 'Henry, 42, M,red,blue',
 'Shaun, 11, M,sky blue,red']

## Lab 5: Putting it all together

In [63]:
src = "/user/student/authors/"
authorNameRDD = sc.textFile(src).map(lambda line: (int(line.split(",")[0]), (line.split(",")[1], line.split(",")[2])))
authorNameRDD.take(5)

[(1, ('Walton', 'Adams')),
 (2, ('Marietta', 'Walsh')),
 (3, ('Lily', 'Wintheiser')),
 (4, ('Estevan', 'Gleason')),
 (5, ('Thaddeus', 'Rowe'))]

In [64]:
src = "/user/student/posts/" 
postsRDD = sc.textFile(src).map(lambda line: (line.split(",")[0], (line.split(",")[1], line.split(",")[2][0:10]))) 
postsRDD.take(5)

[('1', ('1', 'Cupiditate')),
 ('2', ('2', 'Excepturi ')),
 ('3', ('3', 'Enim rerum')),
 ('4', ('4', 'Labore ips')),
 ('5', ('5', 'Placeat ex'))]

In [65]:
import json

def setPhoneName(s):
    if s == "": return "Unknown"
    elif "," in s: return s.replace(",", " or")
    else: return s
    
src = "/user/student/author_phone.json"
phoneRDD = sc.wholeTextFiles(src).flatMap(lambda tup: json.loads(tup[1])).map(lambda kv: (kv.get("author_id", None), kv.get("phone_model", None))).map(lambda kv: (kv[0], setPhoneName(kv[1])))
phoneRDD.take(20)

[(1, 'Samsung Galaxy s21 Ultra'),
 (2, 'Samsung Galaxy A52s'),
 (3, 'Samsung Galaxy Z Fold3'),
 (4, 'Apple iPhone 7'),
 (5, 'Apple iPhone 11'),
 (6, 'Samsung Galaxy A12'),
 (7, 'Samsung Galaxy A52 or Samsung Galaxy s21 Ultra'),
 (8, 'Samsung Galaxy A52s'),
 (9, 'Samsung Galaxy Note 20 or Samsung Galaxy Note20'),
 (10, 'Unknown'),
 (11, 'Samsung Galaxy S8 or Apple iPhone 5'),
 (12, 'Unknown'),
 (13, 'Unknown'),
 (14, 'Samsung Galaxy Z Fold3 or Samsung Galaxy A22'),
 (15, 'Apple iPhone 8'),
 (16, 'Unknown'),
 (17, 'Unknown'),
 (18, 'Unknown'),
 (19, 'Apple iPhone 11 or Samsung Galaxy A52s'),
 (20, 'Samsung Galaxy A12 or Apple iPhone 6')]

In [66]:
import xml.etree.ElementTree as ET
def getPosts(s):
    posts = ET.fromstring(s)
    return posts.iter("record")

def getPostID(elem):
    return elem.find("post_id").text

def getPostLocation(elem):
    return elem.find("location").text

In [67]:
latlongRDD = sc.wholeTextFiles("post_records/*").flatMap(lambda xmls: getPosts(xmls[1])).map(lambda element: (getPostID(element), tuple(getPostLocation(element).split(","))))
latlongRDD.take(5)

[('1', ('-78.67343', ' 146.15251')),
 ('2', ('-24.8449', ' 71.73862')),
 ('3', ('33.15167', ' 90.2584')),
 ('4', ('78.53576', ' -113.2306')),
 ('5', ('37.09904', ' 141.78509'))]

In [68]:
authorPhoneRDD = authorNameRDD.join(phoneRDD)
authorPhoneRDD.take(5)

[(5, (('Thaddeus', 'Rowe'), 'Apple iPhone 11')),
 (10, (('Dewitt', 'Smitham'), 'Unknown')),
 (15, (('Brenda', 'Mayer'), 'Apple iPhone 8')),
 (20, (('Wilfredo', 'Yundt'), 'Samsung Galaxy A12 or Apple iPhone 6')),
 (25, (('Westley', 'Rempel'), 'Unknown'))]

In [69]:
authorPhoneRDD.count()

10000

In [70]:
authorNamePhoneRDD = authorPhoneRDD.map(lambda tup: (tup[0], [tup[1][0][0], tup[1][0][1], tup[1][1]]))
authorNamePhoneRDD.take(5)

[(5, ['Thaddeus', 'Rowe', 'Apple iPhone 11']),
 (10, ['Dewitt', 'Smitham', 'Unknown']),
 (15, ['Brenda', 'Mayer', 'Apple iPhone 8']),
 (20, ['Wilfredo', 'Yundt', 'Samsung Galaxy A12 or Apple iPhone 6']),
 (25, ['Westley', 'Rempel', 'Unknown'])]

In [71]:
postLocationRDD = postsRDD.join(latlongRDD)
postLocationRDD.take(5)

[('5', (('5', 'Placeat ex'), ('37.09904', ' 141.78509'))),
 ('7', (('7', 'Est et aut'), ('-87.08754', ' 8.5678'))),
 ('10', (('10', 'Voluptatum'), ('58.99355', ' -136.22299'))),
 ('38', (('38', 'Dolorum su'), ('-25.42995', ' 160.03288'))),
 ('39', (('39', 'Sed et neq'), ('52.17991', ' -75.04601')))]

In [72]:
postLocationRDD.count()

110000

In [73]:
authorPostLocationRDD = postLocationRDD.map(lambda tup: (int(tup[1][0][0]), [tup[1][0][1], tup[1][1][0], tup[1][1][1]]))
authorPostLocationRDD.take(5)

[(5, ['Placeat ex', '37.09904', ' 141.78509']),
 (7, ['Est et aut', '-87.08754', ' 8.5678']),
 (10, ['Voluptatum', '58.99355', ' -136.22299']),
 (38, ['Dolorum su', '-25.42995', ' 160.03288']),
 (39, ['Sed et neq', '52.17991', ' -75.04601'])]

In [74]:
nameTitlePhoneLocRDD = authorNamePhoneRDD.join(authorPostLocationRDD).map(lambda tup:
                                                                         (tup[1][0][0]+" "+tup[1][0][1]+" on "+tup[1][0][2], "Posted"+tup[1][1][0]+" from lat: "+tup[1][1][1]+" lon:"+tup[1][1][2]))
nameTitlePhoneLocRDD.take(5)

[('Dewitt Smitham on Unknown',
  'PostedVoluptatum from lat: 58.99355 lon: -136.22299'),
 ('Dewitt Smitham on Unknown',
  'PostedMolestiae  from lat: 37.00337 lon: -96.54637'),
 ('Dewitt Smitham on Unknown',
  'PostedConsequatu from lat: -22.20996 lon: 33.00386'),
 ('Dewitt Smitham on Unknown',
  'PostedEarum et n from lat: 37.20708 lon: 82.51313'),
 ('Dewitt Smitham on Unknown',
  'PostedIllo est s from lat: 0.64591 lon: 36.41961')]

In [75]:
zeroValue = []

def seq0p(accumulator, element):
    accumulator.append(element)
    return accumulator

def comb0p(accumulator1, accumulator2):
    return accumulator1 + accumulator2

finalRDD = nameTitlePhoneLocRDD.aggregateByKey(zeroValue, seq0p, comb0p)

finalRDD.take(5)

[('Leann Sipes on Unknown',
  ['PostedAutem repe from lat: 48.17215 lon: 170.37342',
   'PostedAmet exerc from lat: -18.87327 lon: 102.08622',
   'PostedVero labor from lat: 5.9725 lon: -107.00591',
   'PostedUt provide from lat: 49.11693 lon: 78.53206',
   'PostedAmet est e from lat: 70.56624 lon: -13.75885',
   'PostedMaxime dol from lat: -41.31588 lon: 144.69196',
   'PostedEst unde d from lat: -62.57804 lon: -20.393',
   'PostedVoluptatem from lat: 52.89356 lon: 144.90562',
   'PostedVoluptas s from lat: -57.34797 lon: 93.6419',
   'PostedAut nihil  from lat: 43.12988 lon: -117.89295',
   'PostedEt a vitae from lat: -58.66157 lon: -116.0537']),
 ('Alba Greenholt on Samsung Galaxy Note10',
  ['PostedQuae assum from lat: -57.43299 lon: 78.71756',
   'PostedRatione do from lat: -7.19581 lon: 113.39767',
   'PostedCupiditate from lat: 28.14646 lon: -168.70202',
   'PostedSunt aut s from lat: 65.95545 lon: 62.4228',
   'PostedQuia dolor from lat: -81.12062 lon: 5.79223',
   'PostedLiber

## Lab 6: Working with the DataFrame API

In [77]:
jsonDF = spark.read.json("people.json")
jsonDF.printSchema()
jsonDF.show(5)

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



+----+-------+
| age|   name|
+----+-------+
|null|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [78]:
olderDF = jsonDF.where("age >=30")
olderDF.printSchema()
olderDF.show(5)

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [79]:
nameDF = olderDF.select("name")
nameDF.printSchema()
nameDF.show(5)

root
 |-- name: string (nullable = true)

+----+
|name|
+----+
|Andy|
+----+



In [80]:
workerDF = spark.read.csv("people.csv")
workerDF.printSchema()
workerDF.show()

root
 |-- _c0: string (nullable = true)

+------------------+
|               _c0|
+------------------+
|      name;age;job|
|Jorge;30;Developer|
|  Bob;32;Developer|
+------------------+



In [81]:
workerDF = spark.read.option("sep",";").option("header", "true").csv("people.csv")
workerDF.printSchema()
workerDF.show()

root
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- job: string (nullable = true)

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [82]:
usersDF = spark.read.load("users.parquet")
usersDF.printSchema()
usersDF.show()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          null|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



In [83]:
avroDF = spark.read.format("avro").load("users.avro")
avroDF.printSchema()
avroDF.show()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          null|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



In [84]:
avroDF.write.json("/user/student/users_json")

In [85]:
jsonDF.write \
.option("sep","|") \
.option("header","true") \
.csv("/user/student/people_csv")

In [86]:
workerDF.write \
.option("header","false") \
.csv("/user/student/workers")

In [87]:
new_workers = [("Henry", '18', "Mail Clerk"),
("Sharon", '24', "Marketing"),
("Shaun", '32', "Attorney")]

In [88]:
newWorkersDF = spark.createDataFrame(new_workers)
newWorkersDF.show()

+------+---+----------+
|    _1| _2|        _3|
+------+---+----------+
| Henry| 18|Mail Clerk|
|Sharon| 24| Marketing|
| Shaun| 32|  Attorney|
+------+---+----------+



In [89]:
newWorkersDF.write \
.option("header","false") \
.mode("append") \
.csv("/user/student/workers")

In [90]:
finalDF = spark.read.csv("/user/student/workers")

finalDF.show()

+------+---+----------+
|   _c0|_c1|       _c2|
+------+---+----------+
| Henry| 18|Mail Clerk|
|Sharon| 24| Marketing|
| Shaun| 32|  Attorney|
| Jorge| 30| Developer|
|   Bob| 32| Developer|
+------+---+----------+



## Lab 7: Working with Hive from Spark

In [93]:
authorsDF = spark.read.table("mydb.authors")
authorsDF.printSchema()
authorsDF.show(5)

root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- added: string (nullable = true)



+---+----------+----------+--------------------+----------+--------------------+
| id|first_name| last_name|               email| birthdate|               added|
+---+----------+----------+--------------------+----------+--------------------+
|  1|    Walton|     Adams|barmstrong@exampl...|1989-03-01|1997-01-02 04:18:...|
|  2|  Marietta|     Walsh|hand.stella@examp...|2018-05-30|2010-08-26 18:20:...|
|  3|      Lily|Wintheiser|darren.blanda@exa...|1981-08-21|1973-06-11 07:28:...|
|  4|   Estevan|   Gleason|shanahan.aliyah@e...|2013-07-17|1995-01-29 16:08:...|
|  5|  Thaddeus|      Rowe|bednar.robin@exam...|2019-02-26|2017-01-05 04:13:...|
+---+----------+----------+--------------------+----------+--------------------+
only showing top 5 rows



In [94]:
bdayDF = authorsDF.select("id", "email", "birthdate")
bdayDF.printSchema()

root
 |-- id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- birthdate: string (nullable = true)



In [95]:
bdayDF.show(5, truncate=False)

+---+---------------------------+----------+
|id |email                      |birthdate |
+---+---------------------------+----------+
|1  |barmstrong@example.com     |1989-03-01|
|2  |hand.stella@example.net    |2018-05-30|
|3  |darren.blanda@example.org  |1981-08-21|
|4  |shanahan.aliyah@example.net|2013-07-17|
|5  |bednar.robin@example.net   |2019-02-26|
+---+---------------------------+----------+
only showing top 5 rows



In [97]:
bdayDF.write.saveAsTable("mydb.author_bday")

AnalysisException: Table `mydb`.`author_bday` already exists.

In [98]:
nameDF = authorsDF.select("id", "first_name", "last_name")
nameDF.printSchema()
nameDF.show(5)

root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)



+---+----------+----------+
| id|first_name| last_name|
+---+----------+----------+
|  1|    Walton|     Adams|
|  2|  Marietta|     Walsh|
|  3|      Lily|Wintheiser|
|  4|   Estevan|   Gleason|
|  5|  Thaddeus|      Rowe|
+---+----------+----------+
only showing top 5 rows



In [99]:
nameDF.write \
.format("csv") \
.option("path", "/user/student/author_names") \
.option("sep", "\t") \
.option("header", "true") \
.saveAsTable("mydb.author_names")

2026-06-05 02:33:24,659 WARN hive.HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `mydb`.`author_names` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


## Lab 8: Spark SQL Transformations

In [101]:
authorsDF = spark.read.table("mydb.authors")
authorsDF.show(5)

+---+----------+----------+--------------------+----------+--------------------+
| id|first_name| last_name|               email| birthdate|               added|
+---+----------+----------+--------------------+----------+--------------------+
|  1|    Walton|     Adams|barmstrong@exampl...|1989-03-01|1997-01-02 04:18:...|
|  2|  Marietta|     Walsh|hand.stella@examp...|2018-05-30|2010-08-26 18:20:...|
|  3|      Lily|Wintheiser|darren.blanda@exa...|1981-08-21|1973-06-11 07:28:...|
|  4|   Estevan|   Gleason|shanahan.aliyah@e...|2013-07-17|1995-01-29 16:08:...|
|  5|  Thaddeus|      Rowe|bednar.robin@exam...|2019-02-26|2017-01-05 04:13:...|
+---+----------+----------+--------------------+----------+--------------------+
only showing top 5 rows



In [103]:
authorsDF = spark.read.table("mydb.authors").select("first_name", "email", "birthdate")
authorsDF.show(5)

+----------+--------------------+----------+
|first_name|               email| birthdate|
+----------+--------------------+----------+
|    Walton|barmstrong@exampl...|1989-03-01|
|  Marietta|hand.stella@examp...|2018-05-30|
|      Lily|darren.blanda@exa...|1981-08-21|
|   Estevan|shanahan.aliyah@e...|2013-07-17|
|  Thaddeus|bednar.robin@exam...|2019-02-26|
+----------+--------------------+----------+
only showing top 5 rows



In [102]:
authorsDF = spark.read.table("mydb.authors").select("first_name", "email", "birthdate").orderBy("birthdate")
authorsDF.show(5)

+----------+--------------------+----------+
|first_name|               email| birthdate|
+----------+--------------------+----------+
|  Roderick|bobby.conroy@exam...|1970-01-01|
|   Jeffery| verla17@example.org|1970-01-02|
|   Mikayla|  xwalsh@example.org|1970-01-05|
|       Lew|  emie19@example.net|1970-01-06|
|   Shannon|michele48@example...|1970-01-10|
+----------+--------------------+----------+
only showing top 5 rows



In [104]:
authorsDF = spark.read.table("mydb.authors").select("first_name", "email", "birthdate").orderBy("birthdate").limit(3)
authorsDF.show(5)

+----------+--------------------+----------+
|first_name|               email| birthdate|
+----------+--------------------+----------+
|  Roderick|bobby.conroy@exam...|1970-01-01|
|   Jeffery| verla17@example.org|1970-01-02|
|   Mikayla|  xwalsh@example.org|1970-01-05|
+----------+--------------------+----------+



In [105]:
authorsDF = spark.read.table("mydb.authors").select("first_name", "email", "birthdate").where("birthdate>='2000-01-01'").orderBy("birthdate")
authorsDF.show(5)

+----------+--------------------+----------+
|first_name|               email| birthdate|
+----------+--------------------+----------+
|Earnestine|junius98@example.net|2000-01-01|
|      Zoie|    jorn@example.org|2000-01-04|
|  Izabella|xmacejkovic@examp...|2000-01-07|
|    Maryse|nella.grant@examp...|2000-01-07|
|   Xzavier|josefina.lynch@ex...|2000-01-10|
+----------+--------------------+----------+
only showing top 5 rows



In [106]:
authorsDF = spark.read.table("mydb.authors").select("first_name", "email", "birthdate").where("birthdate>='2000-01-01'").orderBy("birthdate").limit(3)
authorsDF.show(5)

+----------+--------------------+----------+
|first_name|               email| birthdate|
+----------+--------------------+----------+
|Earnestine|junius98@example.net|2000-01-01|
|      Zoie|    jorn@example.org|2000-01-04|
|  Izabella|xmacejkovic@examp...|2000-01-07|
+----------+--------------------+----------+



In [108]:
salesDF = spark.read.csv("hdfs:///user/student/sales.csv", header=True, inferSchema=True)

In [109]:
salesDF.printSchema()
salesDF.show(5)

root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- ItemType: string (nullable = true)
 |-- SalesChannel: string (nullable = true)
 |-- OrderPriority: string (nullable = true)
 |-- UnitsSold: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitCost: double (nullable = true)
 |-- TotalRevenue: double (nullable = true)
 |-- TotalCost: double (nullable = true)
 |-- TotalProfit: double (nullable = true)



+--------------------+-------+----------+------------+-------------+---------+---------+--------+------------+----------+-----------+
|              Region|Country|  ItemType|SalesChannel|OrderPriority|UnitsSold|UnitPrice|UnitCost|TotalRevenue| TotalCost|TotalProfit|
+--------------------+-------+----------+------------+-------------+---------+---------+--------+------------+----------+-----------+
|Middle East and N...|  Libya| Cosmetics|     Offline|            M|     8446|    437.2|  263.33|   3692591.2|2224085.18| 1468506.02|
|       North America| Canada|Vegetables|      Online|            M|     3018|   154.06|   90.93|   464953.08| 274426.74|  190526.34|
|Middle East and N...|  Libya| Baby Food|     Offline|            C|     1517|   255.28|  159.42|   387259.76| 241840.14|  145419.62|
|                Asia|  Japan|    Cereal|     Offline|            C|     3322|    205.7|  117.11|    683335.4| 389039.42|  294295.98|
|  Sub-Saharan Africa|   Chad|    Fruits|     Offline|        

In [113]:
salesDF = salesDF.select(
    salesDF.Country,
    salesDF.ItemType,
    salesDF.UnitsSold,
    salesDF.UnitPrice,
    salesDF.UnitCost
)

salesDF.printSchema()
salesDF.show(5)

root
 |-- Country: string (nullable = true)
 |-- ItemType: string (nullable = true)
 |-- UnitsSold: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitCost: double (nullable = true)



+-------+----------+---------+---------+--------+
|Country|  ItemType|UnitsSold|UnitPrice|UnitCost|
+-------+----------+---------+---------+--------+
|  Libya| Cosmetics|     8446|    437.2|  263.33|
| Canada|Vegetables|     3018|   154.06|   90.93|
|  Libya| Baby Food|     1517|   255.28|  159.42|
|  Japan|    Cereal|     3322|    205.7|  117.11|
|   Chad|    Fruits|     9845|     9.33|    6.92|
+-------+----------+---------+---------+--------+
only showing top 5 rows



In [115]:
calcRevDF = salesDF.select(salesDF.Country, salesDF.UnitsSold.cast("float")*salesDF.UnitPrice.cast("float"))
calcRevDF.printSchema()
calcRevDF.show(5)

root
 |-- Country: string (nullable = true)
 |-- (CAST(UnitsSold AS FLOAT) * CAST(UnitPrice AS FLOAT)): float (nullable = true)

+-------+-----------------------------------------------------+
|Country|(CAST(UnitsSold AS FLOAT) * CAST(UnitPrice AS FLOAT))|
+-------+-----------------------------------------------------+
|  Libya|                                            3692591.2|
| Canada|                                            464953.06|
|  Libya|                                            387259.75|
|  Japan|                                             683335.4|
|   Chad|                                             91853.85|
+-------+-----------------------------------------------------+
only showing top 5 rows



In [116]:
calcRevenue = salesDF.UnitsSold.cast("float")*salesDF.UnitPrice.cast("float")
print(calcRevenue)

Column<'(CAST(UnitsSold AS FLOAT) * CAST(UnitPrice AS FLOAT))'>


In [117]:
calcRevDF=salesDF.select(salesDF.Country, calcRevenue.alias("Sales_Revenue"))
calcRevDF.printSchema()
calcRevDF.show(5)

root
 |-- Country: string (nullable = true)
 |-- Sales_Revenue: float (nullable = true)

+-------+-------------+
|Country|Sales_Revenue|
+-------+-------------+
|  Libya|    3692591.2|
| Canada|    464953.06|
|  Libya|    387259.75|
|  Japan|     683335.4|
|   Chad|     91853.85|
+-------+-------------+
only showing top 5 rows



In [118]:
salesDF = spark.read.option("header","true").csv("sales.csv")
calcRevenue = salesDF.UnitsSold.cast("float")*salesDF.UnitPrice.cast("float")
revItemCountryDF = salesDF.select(salesDF.Country, salesDF.ItemType,calcRevenue.alias("Sales Revenue"))
revItemCountryDF.printSchema()
revItemCountryDF.show(5)

root
 |-- Country: string (nullable = true)
 |-- ItemType: string (nullable = true)
 |-- Sales Revenue: float (nullable = true)

+-------+----------+-------------+
|Country|  ItemType|Sales Revenue|
+-------+----------+-------------+
|  Libya| Cosmetics|    3692591.2|
| Canada|Vegetables|    464953.06|
|  Libya| Baby Food|    387259.75|
|  Japan|    Cereal|     683335.4|
|   Chad|    Fruits|     91853.85|
+-------+----------+-------------+
only showing top 5 rows



In [121]:
quota = 3000000

quotaMet = (revItemCountryDF["Sales Revenue"] > quota)

revItemCountryQuotaDF = revItemCountryDF.withColumn("Quota_Met", quotaMet)

revItemCountryQuotaDF.show(5)

+-------+----------+-------------+---------+
|Country|  ItemType|Sales Revenue|Quota_Met|
+-------+----------+-------------+---------+
|  Libya| Cosmetics|    3692591.2|     true|
| Canada|Vegetables|    464953.06|    false|
|  Libya| Baby Food|    387259.75|    false|
|  Japan|    Cereal|     683335.4|    false|
|   Chad|    Fruits|     91853.85|    false|
+-------+----------+-------------+---------+
only showing top 5 rows



In [122]:
# Read the CSV data
salesDF = spark. read.option("header","true").csv("sales.csv").select("Country", "ItemType", "UnitsSold", "UnitPrice")
# set the quota constants
amtQuota = 3000000
cntQuota = 5000
# Create the conditions
salesRevenue = salesDF.UnitsSold * salesDF.UnitPrice
quotaMet = (salesRevenue > amtQuota) | (salesDF.UnitsSold > cntQuota)
# Create the new dataframe
revItemCountryQuotaDF = salesDF.withColumn("Sales_Revenue", salesRevenue).withColumn("Quota_Met", quotaMet)
revItemCountryQuotaDF.printSchema()
revItemCountryQuotaDF.show(5)

root
 |-- Country: string (nullable = true)
 |-- ItemType: string (nullable = true)
 |-- UnitsSold: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- Sales_Revenue: double (nullable = true)
 |-- Quota_Met: boolean (nullable = true)

+-------+----------+---------+---------+------------------+---------+
|Country|  ItemType|UnitsSold|UnitPrice|     Sales_Revenue|Quota_Met|
+-------+----------+---------+---------+------------------+---------+
|  Libya| Cosmetics|     8446|    437.2|3692591.1999999997|     true|
| Canada|Vegetables|     3018|   154.06|         464953.08|    false|
|  Libya| Baby Food|     1517|   255.28|         387259.76|    false|
|  Japan|    Cereal|     3322|    205.7| 683335.3999999999|    false|
|   Chad|    Fruits|     9845|     9.33|          91853.85|     true|
+-------+----------+---------+---------+------------------+---------+
only showing top 5 rows



In [123]:
spark.read.option("header", "true").csv("sales.csv").select("ItemType").groupBy("ItemType").count().show()

+---------------+-----+
|       ItemType|count|
+---------------+-----+
|      Baby Food|   87|
|         Cereal|   79|
|           Meat|   78|
|      Household|   77|
|     Vegetables|   97|
|      Beverages|  101|
|Office Supplies|   89|
|      Cosmetics|   75|
|  Personal Care|   87|
|         Fruits|   70|
|         Snacks|   82|
|        Clothes|   78|
+---------------+-----+



In [124]:
salesDF = spark.read.option("header", "true").csv("sales.csv")
aggDF = salesDF.select("ItemType", salesDF.UnitsSold.cast("int")).groupBy("ItemType").mean("UnitsSold")
aggDF.printSchema()
aggDF.show()

root
 |-- ItemType: string (nullable = true)
 |-- avg(UnitsSold): double (nullable = true)



+---------------+-----------------+
|       ItemType|   avg(UnitsSold)|
+---------------+-----------------+
|      Baby Food|5018.597701149425|
|         Cereal|4908.215189873417|
|           Meat|5229.679487179487|
|      Household|4818.077922077922|
|     Vegetables|4858.515463917526|
|      Beverages|4999.059405940594|
|Office Supplies|4994.179775280899|
|      Cosmetics|          5680.96|
|  Personal Care|5468.091954022989|
|         Fruits|5073.214285714285|
|         Snacks|4818.829268292683|
|        Clothes|4845.974358974359|
+---------------+-----------------+



In [148]:
cntQuotaDict = {"Baby Food": 5018,
"Cereal": 4908,
"Household": 5229,
"Vegetables": 4818,
"Beverages": 4858,
"Office Supplies": 4994,
"Cosmetics": 5680,
"Personal Care": 5468,
"Fruits": 5073,
"Snacks": 4818,
"Clothes": 4845}

In [149]:
def cntQuota(item, cnt):
    meet_this = cntQuotaDict.get(item, None)
    if meet_this == None: meet_this = 0
    my_cnt = int(cnt)
    return my_cnt > meet_this

from pyspark.sql.functions import udf, col
from pyspark.sql.types import BooleanType
quotaUDF = udf(lambda item, cnt: cntQuota(item, cnt), BooleanType())

In [150]:
quotaDF = spark.read.option("header", "true").csv("sales.csv").select("Country", quotaUDF(col("ItemType"), col("UnitsSold")).alias("Met_Quota"))
quotaDF.printSchema()
quotaDF.show()

root
 |-- Country: string (nullable = true)
 |-- Met_Quota: boolean (nullable = true)



+----------+---------+
|   Country|Met_Quota|
+----------+---------+
|     Libya|     true|
|    Canada|    false|
|     Libya|    false|
|     Japan|    false|
|      Chad|     true|
|   Armenia|     true|
|   Eritrea|    false|
|Montenegro|     true|
|   Jamaica|    false|
|      Fiji|    false|
|      Togo|    false|
|Montenegro|    false|
|    Greece|    false|
|     Sudan|    false|
|  Maldives|     true|
|Montenegro|    false|
|   Estonia|    false|
| Greenland|    false|
|Cape Verde|    false|
|   Senegal|     true|
+----------+---------+
only showing top 20 rows



## Lab 9: Working with Spark SQL

In [138]:
authorsDF = spark.sql("SELECT first_name, last_name, email FROM mydb.authors WHERE email LIKE '%org' LIMIT 10")
authorsDF.printSchema()
authorsDF.show(truncate=False)

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)

+----------+----------+---------------------------+
|first_name|last_name |email                      |
+----------+----------+---------------------------+
|Lily      |Wintheiser|darren.blanda@example.org  |
|Dewitt    |Smitham   |yfadel@example.org         |
|Willard   |Wilderman |virgil45@example.org       |
|Jazlyn    |Osinski   |philip.schaden@example.org |
|America   |Marquardt |ulockman@example.org       |
|Alvis     |Crist     |kennith25@example.org      |
|Thurman   |Lesch     |mauricio.harris@example.org|
|Tad       |Bechtelar |jada.grant@example.org     |
|Dudley    |Kirlin    |lauretta82@example.org     |
|Marcelino |Tremblay  |lenora.fritsch@example.org |
+----------+----------+---------------------------+



In [142]:
authorsDF = spark.sql("SELECT first_name, last_name, birthdate FROM mydb.authors WHERE birthdate >= '2000-01-01' ORDER BY birthdate LIMIT 10")
authorsDF.printSchema()
authorsDF.show(truncate=False)

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- birthdate: string (nullable = true)



+----------+----------+----------+
|first_name|last_name |birthdate |
+----------+----------+----------+
|Earnestine|Thiel     |2000-01-01|
|Zoie      |Gusikowski|2000-01-04|
|Maryse    |West      |2000-01-07|
|Izabella  |Hilll     |2000-01-07|
|Xzavier   |Flatley   |2000-01-10|
|Eva       |Beatty    |2000-01-15|
|Roma      |Torp      |2000-01-17|
|Bailee    |Pacocha   |2000-01-17|
|Mariana   |Cremin    |2000-01-19|
|Gustave   |Rippin    |2000-01-19|
+----------+----------+----------+



In [147]:
authorsDF = spark.sql("SELECT * FROM mydb.authors")
authorPhoneDF = spark.read.json("/user/student/author_phone.json")

authorsDF.createOrReplaceTempView("author")
authorPhoneDF.createOrReplaceTempView("author_phone")

spark.sql("""
SELECT a.first_name, a.last_name, b.phone_model
FROM author a
JOIN author_phone b
ON a.id = b.author_id
LIMIT 10
""").show()

+----------+----------+--------------------+
|first_name| last_name|         phone_model|
+----------+----------+--------------------+
|    Walton|     Adams|Samsung Galaxy s2...|
|  Marietta|     Walsh| Samsung Galaxy A52s|
|      Lily|Wintheiser|Samsung Galaxy Z ...|
|   Estevan|   Gleason|      Apple iPhone 7|
|  Thaddeus|      Rowe|     Apple iPhone 11|
|    Cortez|    Russel|  Samsung Galaxy A12|
|     Deion|     Yundt|Samsung Galaxy A5...|
|  Caterina|Cartwright| Samsung Galaxy A52s|
|     Rylee|     Morar|Samsung Galaxy No...|
|    Dewitt|   Smitham|                    |
+----------+----------+--------------------+



In [158]:
spark.sql("""
SELECT a.first_name, a.last_name, b.phone_model, a.birthdate
FROM author a
JOIN author_phone b ON a.id = b.author_id
WHERE b.phone_model = ''
ORDER BY a.birthdate DESC
LIMIT 10
""").show(truncate=False)

+----------+---------+-----------+----------+
|first_name|last_name|phone_model|birthdate |
+----------+---------+-----------+----------+
|Kristin   |Murray   |           |2019-04-18|
|Ladarius  |Parker   |           |2019-04-17|
|Elliot    |Miller   |           |2019-04-08|
|Cassandra |Sporer   |           |2019-04-07|
|Howell    |Hane     |           |2019-04-03|
|Virginia  |Murray   |           |2019-04-02|
|Freda     |Pacocha  |           |2019-03-26|
|Brandon   |Donnelly |           |2019-03-14|
|Julianne  |Schaefer |           |2019-03-11|
|Virgie    |Wyman    |           |2019-03-10|
+----------+---------+-----------+----------+



In [152]:
!pip install sparksql-magic

Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement sparksql-magic (from versions: none)
ERROR: No matching distribution found for sparksql-magic


In [ ]:
%load_ext sparksql_magic

In [151]:
%%sparksql
SELECT a.first_name, a.last_name, b.phone_model, a.birthdate
FROM author a
JOIN author_phone b ON a.id = b.author_id
WHERE b.phone_model = ''
ORDER BY a.birthdate DESC
LIMIT 10

UsageError: Cell magic `%%sparksql` not found.


In [153]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|     mydb|
+---------+



In [154]:
spark.sql("USE mydb")
spark.sql("SHOW TABLES").show()

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
|    mydb| author_bday|      false|
|    mydb|author_names|      false|
|    mydb|     authors|      false|
|        |      author|       true|
|        |author_phone|       true|
+--------+------------+-----------+



In [155]:
spark.sql("CREATE TABLE spark_test (name string, age int)")

2026-06-05 04:10:00,372 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
2026-06-05 04:10:01,486 WARN metastore.HiveMetaStore: Location: hdfs://localhost:9000/user/hive/warehouse/mydb.db/spark_test specified for non-external table:spark_test


DataFrame[]

In [156]:
spark.sql("DESC FORMATTED spark_test").show(truncate=False)

+----------------------------+------------------------------------------------------------+-------+
|col_name                    |data_type                                                   |comment|
+----------------------------+------------------------------------------------------------+-------+
|name                        |string                                                      |null   |
|age                         |int                                                         |null   |
|                            |                                                            |       |
|# Detailed Table Information|                                                            |       |
|Database                    |mydb                                                        |       |
|Table                       |spark_test                                                  |       |
|Owner                       |student                                                     |       |


In [157]:
spark.sql(""" INSERT INTO spark_test VALUES ("Henry", 42) """)
spark.sql(""" INSERT INTO spark_test VALUES ("Jessica", 16) """)
spark.sql("SELECT * FROM spark_test").show()

+-------+---+
|   name|age|
+-------+---+
|  Henry| 42|
|Jessica| 16|
+-------+---+



In [159]:
spark.sql(""" ALTER TABLE spark_test ADD COLUMNS (gender string) """)

DataFrame[]

In [160]:
spark.sql(""" INSERT INTO spark_test VALUES ("Shaun", 42, "Male") """)
spark.sql("SELECT * FROM spark_test").show()

+-------+---+------+
|   name|age|gender|
+-------+---+------+
|  Henry| 42|  null|
|  Shaun| 42|  Male|
|Jessica| 16|  null|
+-------+---+------+



In [161]:
for db in spark.catalog.listDatabases():
    print(db)

Database(name='default', description='Default Hive database', locationUri='hdfs://localhost:9000/user/hive/warehouse')
Database(name='mydb', description='', locationUri='hdfs://localhost:9000/user/hive/warehouse/mydb.db')


In [162]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|     mydb|
+---------+



In [163]:
spark.catalog.setCurrentDatabase("mydb")
spark.sql("use mydb")

DataFrame[]

In [164]:
for table in spark.catalog.listTables():
    print(table)

Table(name='author_bday', database='mydb', description=None, tableType='MANAGED', isTemporary=False)
Table(name='author_names', database='mydb', description=None, tableType='EXTERNAL', isTemporary=False)
Table(name='authors', database='mydb', description='Imported by sqoop on 2026/06/05 02:03:08', tableType='MANAGED', isTemporary=False)
Table(name='spark_test', database='mydb', description=None, tableType='MANAGED', isTemporary=False)
Table(name='author', database=None, description=None, tableType='TEMPORARY', isTemporary=True)
Table(name='author_phone', database=None, description=None, tableType='TEMPORARY', isTemporary=True)


In [165]:
for col in spark.catalog.listColumns("authors"):
    print(col)

Column(name='id', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False)
Column(name='first_name', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)
Column(name='last_name', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)
Column(name='email', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)
Column(name='birthdate', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)
Column(name='added', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)


## Lab 10: Transforming RDDs to DataFrames

In [167]:
import json

def setPhoneName(s):
    if s == "": return "Unknown"
    elif "," in s: return s.replace(",", " or")
    else: return s

phoneRDD = sc.wholeTextFiles("/user/student/author_phone.json") \
.flatMap(lambda tup: json.loads(tup[1])) \
.map(lambda kv: (kv.get("author_id", None), kv.get("phone_model", ""))) \
.map(lambda tup: (tup[0], setPhoneName(tup[1])))
phoneRDD.take(5)

[(1, 'Samsung Galaxy s21 Ultra'),
 (2, 'Samsung Galaxy A52s'),
 (3, 'Samsung Galaxy Z Fold3'),
 (4, 'Apple iPhone 7'),
 (5, 'Apple iPhone 11')]

In [168]:
schema = "author_id int, phone_model string"
phoneDF = spark.createDataFrame(phoneRDD, schema)
phoneDF.printSchema()
phoneDF.show(5, truncate=False)

root
 |-- author_id: integer (nullable = true)
 |-- phone_model: string (nullable = true)



+---------+------------------------+
|author_id|phone_model             |
+---------+------------------------+
|1        |Samsung Galaxy s21 Ultra|
|2        |Samsung Galaxy A52s     |
|3        |Samsung Galaxy Z Fold3  |
|4        |Apple iPhone 7          |
|5        |Apple iPhone 11         |
+---------+------------------------+
only showing top 5 rows



In [169]:
import xml.etree.ElementTree as ET

def getPosts(s):
    posts = ET.fromstring(s)
    return posts.iter("record")

def getPostID(elem):
    return elem.find("post_id").text

def getPostLocation(elem):
    return elem.find("location").text

In [170]:
latlonRDD = sc.wholeTextFiles("/user/student/post_records/") \
.flatMap(lambda tup: getPosts(tup[1])) \
.map(lambda elem: [getPostID(elem), getPostLocation(elem).split(",")]) \
.map(lambda lst: [lst[0], float(lst[1][0]), float(lst[1][1])])
latlonRDD.take(5)

[['1', -78.67343, 146.15251],
 ['2', -24.8449, 71.73862],
 ['3', 33.15167, 90.2584],
 ['4', 78.53576, -113.2306],
 ['5', 37.09904, 141.78509]]

In [171]:
from pyspark.sql.types import *
schema = StructType([
    StructField("post_id", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True)])

In [172]:
latlonDF = spark.createDataFrame(latlonRDD, schema)
latlonDF.printSchema()
latlonDF.show(5, truncate=False)

root
 |-- post_id: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)



+-------+---------+---------+
|post_id|lat      |lon      |
+-------+---------+---------+
|1      |-78.67343|146.15251|
|2      |-24.8449 |71.73862 |
|3      |33.15167 |90.2584  |
|4      |78.53576 |-113.2306|
|5      |37.09904 |141.78509|
+-------+---------+---------+
only showing top 5 rows



In [173]:
latlonDF.write.saveAsTable("mydb.post_latlon")

In [174]:
spark.sql("SHOW TABLES IN mydb").show()

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
|    mydb| author_bday|      false|
|    mydb|author_names|      false|
|    mydb|     authors|      false|
|    mydb| post_latlon|      false|
|    mydb|  spark_test|      false|
|        |      author|       true|
|        |author_phone|       true|
+--------+------------+-----------+



## Lab 11: Working with the DStream API

In [3]:
from pyspark.streaming import StreamingContext
ssc = StreamingContext(sc, 5)
ssc

In [4]:
host = 'localhost'
port = 44444
aliceDS = ssc.socketTextStream(host, port)

In [ ]:
wcDS = aliceDS.flatMap(lambda line: line.split(" ")).map(lambda word: (word, 1)).reduceByKey(lambda v1, v2: v1 + v2)

In [6]:
wcDS.pprint()

In [7]:
def printCount(t, r):
    print("Word Count at time", t)
    for cnt in r.take(10):
        print(cnt[0], ":", cnt[1])

wcDS.foreachRDD(lambda time, rdd: printCount(time, rdd))

In [ ]:
ssc.start()
ssc.awaitTermination()

2026-06-05 10:24:17,062 WARN storage.RandomBlockReplicationPolicy: Expecting 1 replicas with only 0 peer/s.
2026-06-05 10:24:17,156 WARN storage.BlockManager: Block input-0-1780622656400 replicated to only 0 peer(s) instead of 1 peers
2026-06-05 10:24:17,660 WARN storage.RandomBlockReplicationPolicy: Expecting 1 replicas with only 0 peer/s.
2026-06-05 10:24:17,661 WARN storage.BlockManager: Block input-0-1780622656600 replicated to only 0 peer(s) instead of 1 peers
2026-06-05 10:24:17,762 WARN storage.RandomBlockReplicationPolicy: Expecting 1 replicas with only 0 peer/s.
2026-06-05 10:24:17,763 WARN storage.BlockManager: Block input-0-1780622656800 replicated to only 0 peer(s) instead of 1 peers
2026-06-05 10:24:17,937 WARN storage.RandomBlockReplicationPolicy: Expecting 1 replicas with only 0 peer/s.
2026-06-05 10:24:17,937 WARN storage.BlockManager: Block input-0-1780622657000 replicated to only 0 peer(s) instead of 1 peers
2026-06-05 10:24:18,049 WARN storage.RandomBlockReplicationP

## Lab 12: Working with Multi-Batch DStream API

## Lab 13: Working with Structured Streaming API

In [5]:
postsDF = spark.readStream \
.format("socket") \
.option("host", "localhost") \
.option("port", "44444") \
.load()

2026-06-05 11:46:05,490 WARN sources.TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [6]:
from pyspark.sql.functions import *
aPostDF = postsDF \
.withColumn("id",
split(postsDF.value, ",")[0].cast("integer")) \
.withColumn("author_id",
split(postsDF.value, ",")[1].cast("integer")) \
.withColumn("title",
split(postsDF.value, ",")[2]) \
.withColumn("description",
split(postsDF.value, ",")[3]) \
.withColumn("content",
split(postsDF.value, ",")[4]) \
.withColumn("date",
split(postsDF.value, ",")[5])

In [7]:
myOutDF = aPostDF \
.select("author_id",
aPostDF.title[0:10].alias("Post_Title"),
"date")

In [ ]:
myStream = myOutDF.writeStream \
.format("console") \
.option("truncate","false") \
.outputMode("append") \
.trigger(processingTime="2 seconds") \
.start()
myStream.awaitTermination()

2026-06-05 11:47:34,187 WARN streaming.StreamingQueryManager: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-4a312487-a0f2-4689-bee2-6c37626afd64. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+----------+----+
|author_id|Post_Title|date|
+---------+----------+----+
+---------+----------+----+



2026-06-05 11:48:32,385 WARN streaming.ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000 milliseconds, but spent 48234 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+---------+----------+----------+
|author_id|Post_Title|date      |
+---------+----------+----------+
|1        |Cupiditate|2011-09-29|
|2        |Excepturi |1991-07-30|
|3        |Enim rerum|1997-08-17|
|4        |Labore ips|1979-10-23|
|5        |Placeat ex|1997-04-26|
|6        |Non nihil |2004-10-23|
|7        |Est et aut|1973-12-23|
|8        |Dolorem ra|1971-10-05|
|9        |Eum cumque|1982-12-09|
|10       |Voluptatum|2015-12-08|
|11       |Dolor alia|2005-02-02|
|12       |Dolores vo|2003-04-09|
|13       |Provident |2007-04-09|
|14       |Maxime aut|1975-10-14|
|15       |Rem sint c|1993-04-26|
|16       |Eius ea ha|2018-12-10|
|17       |Sed ex sit|1992-07-03|
|18       |Ea consect|2018-09-24|
|19       |Quis repud|1995-12-12|
|20       |Facere atq|1983-05-12|
+---------+----------+----------+
only showing top 20 rows



2026-06-05 11:48:44,713 WARN streaming.ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000 milliseconds, but spent 12327 milliseconds


...

-------------------------------------------
Batch: 458
-------------------------------------------
+---------+----------+----------+
|author_id|Post_Title|date      |
+---------+----------+----------+
|9669     |Magnam eni|1982-04-27|
|9670     |Rerum mole|2016-11-13|
|9671     |Quia fugit|1985-08-01|
|9672     |Autem nobi|1975-02-03|
|9673     |Temporibus|1970-11-27|
|9674     |Esse moles|1994-07-26|
|9675     |Veniam ape|1981-03-03|
|9676     |Est omnis |2005-05-30|
|9677     |Quis nemo |1996-10-21|
|9678     |Repellendu|2000-12-01|
|9679     |Nesciunt n|1999-02-09|
|9680     |Est volupt|2009-08-16|
|9681     |Facere ips|2003-12-19|
|9682     |Rerum quis|1971-12-19|
|9683     |Enim et et|2005-10-08|
|9684     |Atque odio|1992-06-14|
|9685     |Molestiae |2002-08-16|
|9686     |Non velit |1992-06-24|
|9687     |Culpa fuga|1975-03-16|
|9688     |Tenetur bl|1995-03-24|
+---------+----------+----------+
only showing top 20 rows

-------------------------------------------
Batch: 459
----

2026-06-05 12:06:08,233 WARN streaming.ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000 milliseconds, but spent 2232 milliseconds


-------------------------------------------
Batch: 468
-------------------------------------------
+---------+----------+----------+
|author_id|Post_Title|date      |
+---------+----------+----------+
|44       |Animi modi|2009-09-16|
|45       |Dignissimo|1990-04-25|
|46       |Dolorem et|2010-08-31|
|47       |Corrupti o|1994-06-22|
|48       |Autem alia|1988-06-15|
|49       |Consectetu|1995-07-08|
|50       |Ipsam even|1987-04-01|
|51       |Nihil nihi|1986-09-09|
|52       |Vero omnis|2013-07-14|
|53       |Alias labo|1994-05-29|
|54       |Dolorem si|1973-06-15|
|55       |Minus exce|2011-01-20|
|56       |Et eum vol|1983-12-22|
|57       |Ut accusan|1976-05-21|
|58       |Ex rem et |1990-01-10|
|59       |Excepturi |1976-02-15|
|60       |Vel autem |2008-12-25|
|61       |Quibusdam |1992-06-12|
|62       |Aut delect|1990-05-13|
|63       |Est ut mag|2004-01-19|
+---------+----------+----------+
only showing top 20 rows



2026-06-05 12:06:10,606 WARN streaming.ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000 milliseconds, but spent 2373 milliseconds


-------------------------------------------
Batch: 469
-------------------------------------------
+---------+----------+----------+
|author_id|Post_Title|date      |
+---------+----------+----------+
|85       |Tempore nu|1987-02-04|
|86       |Necessitat|2010-02-28|
|87       |Omnis cupi|1978-08-29|
|88       |Voluptatum|1981-10-08|
|89       |Nam itaque|2003-06-02|
|90       |Dolorem er|2010-08-03|
|91       |Tenetur ea|1978-07-14|
|92       |Non nisi m|1976-11-22|
|93       |Voluptatum|1970-01-05|
|94       |Ut error e|1986-02-04|
|95       |Possimus c|2015-07-20|
|96       |Voluptate |2006-09-20|
|97       |Consequunt|2004-01-11|
|98       |Delectus d|1982-01-21|
|99       |Voluptatum|1970-01-30|
|100      |Ut dolor d|2009-02-23|
|101      |Pariatur e|1993-02-13|
|102      |Soluta quo|1997-08-03|
|103      |Ut enim et|2013-03-18|
|104      |Tempora qu|2006-02-26|
+---------+----------+----------+
only showing top 20 rows

-------------------------------------------
Batch: 470
----

## Lab 14: Create an Apache Spark application